# 02 — Quality control, clustering and cell types

**CIAD single-cell workshop**

Adapted from the Seurat *Guided Clustering Tutorial*:
<https://satijalab.org/seurat/articles/pbmc3k_tutorial>

Keep that page open — everything here maps onto it, so you can go back to the
original after the workshop.

**The data.** 2,700 peripheral blood mononuclear cells (PBMCs) from a healthy
donor, sequenced by 10x Genomics.

**What we do.** Take a matrix of counts with no labels of any kind, and end with
named cell types.

1. quality control — throw out low quality cells 
2. normalization and feature selection
3. PCA, then clustering
4. UMAP, to see the result
5. marker genes, and naming the clusters

**How to run.** *Shift + Enter* runs a cell. Run them in order, top to bottom.

If the runtime disconnects, re-run the setup cell and then the checkpoint cell
in the section you were in — you will not have to start again.

## Setup

This installs Seurat and everything else into the temporary machine Colab gave
you. About a minute the first time, seconds if you run it again.

In [ ]:
# This line installs the packages we need. It is a shortcut for the
# workshop, so that nobody spends the class waiting for an install.
source("https://raw.githubusercontent.com/MartinLoza/CIAD_workshop_sc/main/setup/setup.R")

# This is the normal way to load a package in R. You will write lines
# like these at the top of every script you make.
library(Seurat)
library(ggplot2)
library(dplyr)
library(patchwork)

# Size of every figure in this notebook, in inches. Change these two
# numbers if a plot looks too small or too large.
options(repr.plot.width = 10, repr.plot.height = 7)

## 1. The data

We download the counts from 10x Genomics. Note this is downloaded by the Colab
machine, not by your laptop.

The files are the standard 10x output: three files describing a sparse matrix.

- `matrix.mtx` — the counts themselves
- `barcodes.tsv` — one line per cell
- `genes.tsv` — one line per gene

In [ ]:
url <- "https://cf.10xgenomics.com/samples/cell/pbmc3k/pbmc3k_filtered_gene_bc_matrices.tar.gz"

download.file(url, "pbmc3k.tar.gz", quiet = TRUE)
untar("pbmc3k.tar.gz")

list.files("filtered_gene_bc_matrices/hg19")

### What is inside those files

Before loading anything, look at the files themselves. They are plain text.

In [ ]:
dir <- "filtered_gene_bc_matrices/hg19"

# readLines(n = 3) stops after 3 lines; matrix.mtx has millions of them
cat("barcodes.tsv — one line per cell\n")
writeLines(readLines(file.path(dir, "barcodes.tsv"), n = 3))

cat("\ngenes.tsv — one line per gene\n")
writeLines(readLines(file.path(dir, "genes.tsv"), n = 3))

cat("\nmatrix.mtx — the counts\n")
writeLines(readLines(file.path(dir, "matrix.mtx"), n = 6))

Three things to notice.

- **`barcodes.tsv`** holds cell barcodes, such as `AAACATACAACCAC-1`. A barcode
  is a short DNA sequence that was attached to everything coming from one
  droplet. It is simply the name of the cell.
- **`genes.tsv`** has two columns: the Ensembl gene id, and the gene symbol we
  read (`MS4A1`, `CD3E`). Seurat uses the symbol.
- **`matrix.mtx`** starts with two header lines. The next line gives three
  numbers: how many genes, how many cells, and how many counts are not zero.
  Every line after that is one count, written as *gene number, cell number,
  count*.

Multiply the first two numbers and compare with the third. The full table would
hold about 88 million values, and the file stores only a few million of them.

`Read10X()` reads those three files into one sparse matrix: genes in rows,
cells in columns.

This is a sparse dataset. Most entries are zero — a given gene is not detected in a given
cell — and storing only the non-zero values is what makes this fit in memory.

In [ ]:
pbmc.data <- Read10X(data.dir = "filtered_gene_bc_matrices/hg19")

cat("genes:", nrow(pbmc.data), "\n")
cat("cells:", ncol(pbmc.data), "\n\n")

# the names come straight from genes.tsv and barcodes.tsv
cat("first genes:", head(rownames(pbmc.data), 4), "\n")
cat("first cells:", head(colnames(pbmc.data), 2), "\n\n")

# a corner of the matrix: "." is a zero that is not stored
pbmc.data[c("CD3D", "TCL1A", "MS4A1"), 1:20]

### The Seurat object

`CreateSeuratObject()` wraps the matrix together with everything we are about to
compute — QC metrics, clusters, UMAP coordinates — in a single object.

Two initial filters are applied here. These filters are not strict so we don't lose many cells or genes here:

- `min.cells = 3` — drop genes seen in fewer than 3 cells. They carry no
  information and only cost memory.
- `min.features = 200` — drop droplets with fewer than 200 genes detected.
  These are almost never intact cells.

In [ ]:
pbmc <- CreateSeuratObject(
  counts       = pbmc.data,
  project      = "pbmc3k",
  min.cells    = 3,
  min.features = 200
)

pbmc

## 2. Quality control

We can't look each cell in a microscope, so we judge them by its counts.
Three numbers help us to assess cells:

| metric | what it is | what an extreme value suggests |
|---|---|---|
| `nFeature_RNA` | genes detected in the cell | very low: empty droplet or dying cell. very high: two cells in one droplet |
| `nCount_RNA` | total molecules in the cell | same as above |
| `percent.mt` | % of counts from mitochondrial genes | high: the cell membrane broke, cytoplasmic RNA leaked out, mitochondrial RNA stayed |

The first two are computed for you by `CreateSeuratObject()`. The third we add
ourselves: human mitochondrial gene symbols all start with `MT-`.

In [ ]:
pbmc[["percent.mt"]] <- PercentageFeatureSet(pbmc, pattern = "^MT-")

head(pbmc@meta.data, 5)

### Looking at the distributions

A violin plot per metric. We are looking for the bulk of the cells, and for the
tails we might want to cut.

In [ ]:
VlnPlot(pbmc,
        features = c("nFeature_RNA", "nCount_RNA", "percent.mt"),
        ncol     = 3)

Metrics are easier to judge two at a time. Read the left panel as "do
high-count cells have high mitochondrial content", and the right as "do
high-count cells have more genes" — the right one should be a tight relationship,
and cells falling off it are suspicious.

In [ ]:
p1 <- FeatureScatter(pbmc, feature1 = "nCount_RNA", feature2 = "percent.mt")
p2 <- FeatureScatter(pbmc, feature1 = "nCount_RNA", feature2 = "nFeature_RNA")

p1 + p2

### Filtering

The tutorial's thresholds: more than 200 and fewer than 2,500 genes, less than
5% mitochondrial content.

These numbers are **not universal**. They come from looking at the plots above,
for this tissue and this protocol. On another dataset you would look again.

In [ ]:
cells_before <- ncol(pbmc)

pbmc <- subset(pbmc,
               subset = nFeature_RNA > 200 &
                        nFeature_RNA < 2500 &
                        percent.mt   < 5)

cat("before:", cells_before, "cells\n")
cat("after :", ncol(pbmc), "cells\n")
cat("removed:", cells_before - ncol(pbmc),
    sprintf("(%.1f%%)\n", 100 * (cells_before - ncol(pbmc)) / cells_before))

### ✏️ Exercise 1

How sensitive is that decision?

Re-run the filter on a copy of the object with a **stricter** mitochondrial cut
of 2.5%, and report how many more cells you lose. Do not overwrite `pbmc`.

Fill in the blanks:

In [ ]:
# strict <- subset(pbmc_unfiltered,
#                  subset = nFeature_RNA > 200 &
#                           nFeature_RNA < 2500 &
#                           percent.mt   < ______)
# ncol(strict)

# Hint: you no longer have the unfiltered object — pbmc was overwritten above.
# What is the cheapest way to get it back?

## 3. Normalisation

### Why we need it

Two cells of the same type can give very different counts. One cell may be
captured better, or sequenced deeper, than another. This is technical. It says
nothing about the biology of the cell.

First look at how large that difference is here. `nCount_RNA` is the total
number of counts in a cell, so it is a measure of sequencing depth.

In [ ]:
depth <- pbmc$nCount_RNA

cat("total counts per cell\n")
cat("  smallest cell:", min(depth), "\n")
cat("  median cell  :", median(depth), "\n")
cat("  largest cell :", max(depth), "\n")
cat("  the largest cell has", round(max(depth) / min(depth), 1),
    "times more counts than the smallest one\n")

The largest cell has many times more counts than the smallest one. If we compare
cells now, we mostly compare sequencing depth, not biology.

`NormalizeData()` removes this. For each cell it does three things:

1. divide every count in the cell by the total counts of that cell
2. multiply by 10,000, a fixed number, so the values are easier to read
3. take `log(x + 1)`

Step 1 removes the depth. Step 3 brings the very large values closer to the rest,
so that a few loud genes do not dominate.

The result goes into a new layer called `data`. The raw counts stay in the
`counts` layer and are not changed.

In [ ]:
pbmc <- NormalizeData(pbmc,
                      normalization.method = "LogNormalize",
                      scale.factor         = 10000)

# raw counts and normalised values now live side by side
Layers(pbmc[["RNA"]])

### One gene, before and after

Numbers are easier to believe when you can see them. We follow one gene through
the normalisation.

`ACTB` (beta-actin) is a good gene for this. It is expressed in every cell, so
every cell has a value.

In [ ]:
gene <- "ACTB"   # beta-actin, expressed in every cell, so every cell has a value

gene_df <- data.frame(
  depth = pbmc$nCount_RNA,
  raw   = as.numeric(LayerData(pbmc, assay = "RNA", layer = "counts")[gene, ]),
  norm  = as.numeric(LayerData(pbmc, assay = "RNA", layer = "data")[gene, ])
)

p1 <- ggplot(gene_df, aes(x = raw)) +
  geom_histogram(bins = 50) +
  labs(title = paste(gene, "before normalisation"),
       x = "raw counts in the cell", y = "number of cells")

p2 <- ggplot(gene_df, aes(x = norm)) +
  geom_histogram(bins = 50) +
  labs(title = paste(gene, "after normalisation"),
       x = "normalised value", y = "number of cells")

p1 + p2

The shape of the distribution changes. The raw counts have a long tail to the
right. The normalised values are more symmetric.

The shape is not the main point though. The main question is this: does the gene
still follow sequencing depth?

In [ ]:
cat("correlation with sequencing depth\n")
cat("  before:", round(cor(gene_df$depth, gene_df$raw),  2), "\n")
cat("  after :", round(cor(gene_df$depth, gene_df$norm), 2), "\n")

p3 <- ggplot(gene_df, aes(x = depth, y = raw)) +
  geom_point(size = 0.3, alpha = 0.3) +
  labs(title = "before", x = "total counts in the cell",
       y = paste(gene, "raw counts"))

p4 <- ggplot(gene_df, aes(x = depth, y = norm)) +
  geom_point(size = 0.3, alpha = 0.3) +
  labs(title = "after", x = "total counts in the cell",
       y = paste(gene, "normalised value"))

p3 + p4

In the left panel the points rise from left to right: cells with more total
counts have more `ACTB` counts. In the right panel that trend is much weaker,
and the correlation number is much smaller.

This is what we wanted. What is left is closer to how much `ACTB` the cell really
expresses, and not how deep it was sequenced.

**Try it:** change `gene` to `"MS4A1"` and run the two cells again. `MS4A1` is a
B cell marker, so most cells do not express it at all. The plots look very
different. Why?

## 4. Variable features

Most genes are expressed at much the same level in every cell. They add noise and
computation without helping to tell cell types apart.

`FindVariableFeatures()` keeps the 2,000 genes whose variance is highest *given
their mean* — the `vst` method — since raw variance would just select the
highly expressed genes.

In [ ]:
pbmc <- FindVariableFeatures(pbmc, selection.method = "vst", nfeatures = 2000)

top10 <- head(VariableFeatures(pbmc), 10)
top10

In [ ]:
p <- VariableFeaturePlot(pbmc)
LabelPoints(plot = p, points = top10, repel = TRUE)

Look at what came out: `PPBP` (platelets), `LYZ` (monocytes), `GNLY` and
`NKG7` (NK cells), `S100A8` (neutrophils and monocytes). The method knows nothing
about immunology, yet the most variable genes are markers of the cell types
present. That is the whole idea.

## 5. Scaling

PCA is driven by variance, so a highly expressed gene would dominate simply by
being highly expressed. `ScaleData()` centres each gene at zero and scales it to
unit variance, so genes are compared on equal terms.

We scale all genes here, which takes a few seconds. The default scales only the
variable features — enough for PCA, but the heatmaps later look better with
everything scaled.

In [ ]:
pbmc <- ScaleData(pbmc, features = rownames(pbmc))

## 6. PCA

2,000 variable genes is still far too many dimensions. PCA compresses them into a
few dozen components that capture the structure, and it is those components —
not the genes — that clustering and UMAP use.

In [ ]:
pbmc <- RunPCA(pbmc, features = VariableFeatures(pbmc), verbose = FALSE)

print(pbmc[["pca"]], dims = 1:5, nfeatures = 5)

Each component is a weighted combination of genes. Printed above are the
five genes pulling hardest on each of the first five components — and they read
like cell type signatures again.

A heatmap makes it concrete. Cells are ordered by their score on the component,
genes by their loading. A clean block structure means the component separates
something real.

In [ ]:
DimHeatmap(pbmc, dims = 1:6, cells = 500, balanced = TRUE)

### How many components?

Keep too few and you lose real structure. Keep too many and you cluster on noise.

The elbow plot shows how much variance each component explains. Where the curve
flattens, the remaining components are mostly noise.

In [ ]:
ElbowPlot(pbmc, ndims = 30)

### ✏️ Exercise 2

Look at the elbow plot and decide where it flattens.

The tutorial uses 10. Is that defensible from the plot? Would 15 change much?

Set `n_dims` below to the number you would defend, and we will use it for the
rest of the notebook.

In [ ]:
n_dims <- ______   # replace with your choice, e.g. 10

cat("using", n_dims, "principal components\n")

## 7. Clustering

Two steps.

`FindNeighbors()` builds a graph: every cell is connected to the cells nearest to
it in PCA space.

`FindClusters()` then finds groups of cells that are more connected to each other
than to the rest — communities in that graph.

`resolution` controls granularity. Higher gives more, smaller clusters. There is
no correct value; 0.4–1.2 is the usual range for a few thousand cells.

In [ ]:
pbmc <- FindNeighbors(pbmc, dims = 1:n_dims)
pbmc <- FindClusters(pbmc, resolution = 0.5)

table(Idents(pbmc))

## 8. UMAP

UMAP places every cell on a 2D map, trying to keep cells that were neighbours in
PCA space as neighbours on the map.

Two warnings, worth saying out loud before anyone over-reads a picture:

- **Distance between clusters means little.** Two clusters far apart on a UMAP
  are not necessarily more different than two that are close.
- **UMAP does not define the clusters.** The clusters were computed in PCA space,
  in `n_dims` dimensions. UMAP only draws them.

In [ ]:
pbmc <- RunUMAP(pbmc, dims = 1:n_dims, verbose = FALSE)

DimPlot(pbmc, reduction = "umap", label = TRUE) + NoLegend()

### 💾 Checkpoint

If you fell behind, or the runtime crashed, this is the point worth saving.

The cell below writes the object to the Colab machine's disk. It disappears when
the runtime shuts down, but it survives you re-running cells by accident.

In [ ]:
saveRDS(pbmc, "pbmc_clustered.rds")

# To come back to this point later:
# pbmc <- readRDS("pbmc_clustered.rds")

cat("saved:", round(file.size("pbmc_clustered.rds") / 1e6, 1), "MB\n")

## 9. Marker genes

We have clusters with numbers. To name them we need to know what each one
expresses that the others do not.

`FindAllMarkers()` compares every cluster against all the remaining cells, one
cluster at a time. `only.pos = TRUE` keeps only genes that are *higher* in the
cluster, which is what you want for naming things.

This is the slowest cell in the notebook — around a minute.

In [ ]:
markers <- FindAllMarkers(pbmc, only.pos = TRUE, verbose = FALSE)

markers %>%
  group_by(cluster) %>%
  slice_max(avg_log2FC, n = 3) %>%
  ungroup() %>%
  as.data.frame()

Two ways to look at a marker: where it is expressed on the map, and how
strongly it is expressed per cluster.

In [ ]:
FeaturePlot(pbmc,
            features = c("MS4A1", "CD3E", "CD14", "FCGR3A",
                         "GNLY", "LYZ", "FCER1A", "PPBP"),
            ncol     = 4)

In [ ]:
VlnPlot(pbmc, features = c("MS4A1", "CD3E", "CD14", "PPBP"), ncol = 2)

A heatmap of the top markers per cluster gives the whole picture at once.
Each column is a cell, grouped by cluster; each row a gene.

In [ ]:
top_markers <- markers %>%
  group_by(cluster) %>%
  dplyr::filter(avg_log2FC > 1) %>%
  slice_head(n = 10) %>%
  ungroup()

DoHeatmap(pbmc, features = top_markers$gene) + NoLegend()

## 10. Naming the clusters

This is the step no algorithm does for you. You match marker genes to known
biology.

For PBMCs the canonical markers are well established:

| markers | cell type |
|---|---|
| `IL7R`, `CCR7` | naive CD4 T |
| `IL7R`, `S100A4` | memory CD4 T |
| `CD14`, `LYZ` | CD14+ monocytes |
| `MS4A1` | B |
| `CD8A` | CD8 T |
| `FCGR3A`, `MS4A7` | FCGR3A+ monocytes |
| `GNLY`, `NKG7` | NK |
| `FCER1A`, `CST3` | dendritic cells |
| `PPBP` | platelets |

### ✏️ Exercise 3

The labels below are the tutorial's, in the tutorial's cluster order. **Your
cluster numbering may differ** — it depends on the number of components and the
resolution you chose.

Check them against your own markers before you accept them. If your cluster 3 is
not the B cell cluster, the vector is wrong for you and needs reordering.

In [ ]:
# The order must match levels(pbmc) — check first:
levels(pbmc)

In [ ]:
new_ids <- c("Naive CD4 T", "CD14+ Mono", "Memory CD4 T", "B",
             "CD8 T", "FCGR3A+ Mono", "NK", "DC", "Platelet")

# only works if you have exactly as many clusters as labels
stopifnot(length(new_ids) == length(levels(pbmc)))

names(new_ids) <- levels(pbmc)
pbmc <- RenameIdents(pbmc, new_ids)

DimPlot(pbmc, reduction = "umap", label = TRUE, pt.size = 0.5) + NoLegend()

Keep the labels somewhere permanent. `Idents()` is easy to overwrite by
accident; a metadata column is not.

In [ ]:
pbmc$cell_type <- Idents(pbmc)

table(pbmc$cell_type)

In [ ]:
saveRDS(pbmc, "pbmc_annotated.rds")

cat("saved:", round(file.size("pbmc_annotated.rds") / 1e6, 1), "MB\n")

## What we did

From a matrix of counts with no labels, to named immune cell types, in about ten
steps.

Worth carrying forward:

- **QC thresholds are judgement, not defaults.** We looked at distributions and
  then chose. Different tissue, different numbers.
- **Clustering happens in PCA space, not on the UMAP.** The picture is a
  projection of the result, not the result.
- **Resolution and dimensionality are choices.** They change how many clusters
  you get. If a conclusion only holds at one resolution, it is not a conclusion.
- **Naming clusters is the biology.** Everything before it is bookkeeping.

Next notebook: what happens when the data comes from more than one sample, and
the batch effect is larger than the biology.

---

### Answers

<details>
<summary>Click to expand</summary>

**Exercise 1**

`pbmc` was overwritten by the filtering step, so the unfiltered object is gone.
The cheapest way back is to rebuild it from `pbmc.data`, which is still in memory:

```r
unf <- CreateSeuratObject(pbmc.data, min.cells = 3, min.features = 200)
unf[["percent.mt"]] <- PercentageFeatureSet(unf, pattern = "^MT-")
strict <- subset(unf, subset = nFeature_RNA > 200 &
                               nFeature_RNA < 2500 &
                               percent.mt   < 2.5)
ncol(strict)
```

The real lesson: assigning a filtered object back onto its own name throws away
the thing you need to check the filter. Use a new name.

**Exercise 2**

Anything from 10 to 15 is defensible. The curve is clearly flat past ~15, and
the difference between 10 and 15 is small — which is the point: if your clusters
change completely between 10 and 15 components, they were never stable.

**Exercise 3**

Compare each cluster's top markers to the table above. With 10 components at
resolution 0.5 you normally get 9 clusters in the tutorial's order, but this is
not guaranteed. If `stopifnot()` fails, you have a different number of clusters —
lower the resolution, or write a label vector matching what you actually have.

</details>